In [1]:
import glob, os, json, time
import numpy as np, pandas as pd
from numpy.lib.format import open_memmap
from concurrent.futures import ThreadPoolExecutor

K_TEST = 30       # patches per test place (same as val)

PL = glob.glob("/kaggle/input/**/v2_places.csv", recursive=True)[0]
IN = os.path.dirname(PL); CONSDIR = os.path.join(IN, "cons")
pl = pd.read_csv(PL)
dirs = {int(os.path.basename(os.path.dirname(p))): os.path.dirname(p)
        for p in glob.glob("/kaggle/input/**/labels.npz", recursive=True)}
print("places:", len(pl), "| dirs:", len(dirs), "| cons dir:", CONSDIR)
OUT = "/kaggle/working"

def lut(d):
    t = np.full(256, 255, np.uint8)
    for k, v in d.items(): t[k] = v
    return t
WC = lut({10:1, 20:5, 30:2, 40:4, 50:6, 60:7, 70:8, 80:0, 90:3, 95:1, 100:7})
ES = lut({1:0, 2:1, 4:3, 5:4, 7:6, 8:7, 9:8, 11:253})   # 253 = rangeland (grass or shrub)
DW = lut({i: i for i in range(9)})
CLS = ["water", "trees", "grass", "flooded", "crops", "shrub", "built", "bare", "snow"]

split = "test"
allids = sorted(pl[pl.split == split].id.tolist())
ids = sorted(pl[(pl.split == split) & (pl.months_ok >= 3)].id.tolist())
print("test places:", len(allids), "| usable (>=3 months):", len(ids), "| excluded:", sorted(set(allids) - set(ids)))
N = len(ids) * K_TEST
X = open_memmap(f"{OUT}/X_test.npy", mode="w+", dtype=np.float16, shape=(N, 6, 256, 256))
Y = open_memmap(f"{OUT}/Y_test.npy", mode="w+", dtype=np.uint8, shape=(N, 4, 256, 256))   # 0 cons, 1 WC, 2 DW, 3 ESRI
V = np.zeros(N, bool); PID = np.zeros(N, np.int32); PP = np.zeros(N, np.int16); TT = np.zeros(N, np.int16)

def work(a):
    si, pid = a
    rng = np.random.default_rng(pid + 5000)
    d = dirs[pid]
    months = {}
    for f in glob.glob(os.path.join(d, "20??-??.npz")):
        b = os.path.basename(f)[:-4]
        months[(int(b[:4]) - 2020) * 12 + int(b[5:7]) - 1] = f
    centers = [t for t in months if (t - 1) in months and (t + 1) in months]
    if not centers: return
    y = np.load(os.path.join(d, "labels.npz"))["y"]
    cons = np.load(f"{CONSDIR}/{pid:04d}.npz")["cons"]
    n = y.shape[0]
    ign = (cons == 255).mean((2, 3))
    cand = [(t, p) for t in centers for p in range(n) if ign[p, 0 if t < 12 else 1] < 0.6]
    if not cand: return
    sel = rng.choice(len(cand), size=min(K_TEST, len(cand)), replace=False)
    cache = {}
    def get(t):
        if t not in cache: cache[t] = np.load(months[t])["x"]
        return cache[t]
    for j, c in enumerate(sel):
        t, p = cand[c]; yi = 0 if t < 12 else 1
        x = np.concatenate([get(t - 1)[p], get(t)[p], get(t + 1)[p]]).astype(np.float32)
        x = np.clip(np.nan_to_num(x, nan=-35, neginf=-35, posinf=5), -35, 5)
        s = si * K_TEST + j
        X[s] = x.astype(np.float16)
        Y[s, 0] = cons[p, yi]
        Y[s, 1] = WC[y[p, yi]]
        Y[s, 2] = DW[y[p, 4 + yi]]
        Y[s, 3] = ES[y[p, 2 + yi]]
        V[s] = True; PID[s] = pid; PP[s] = p; TT[s] = t

t0 = time.time()
with ThreadPoolExecutor(4) as ex:
    for i, _ in enumerate(ex.map(work, enumerate(ids))):
        if i % 20 == 0: print(split, i, "/", len(ids), f"{time.time() - t0:.0f}s", flush=True)
X.flush(); Y.flush()
np.savez(f"{OUT}/meta_test.npz", valid=V, pid=PID, pp=PP, tt=TT)

vi = np.where(V)[0]
print(f"\n{split}: valid samples {len(vi)} of {N} | places {len(set(PID[vi].tolist()))} of {len(ids)}")
cnt = np.zeros((4, 256))
for s in range(0, len(vi), 500):
    yy = Y[vi[s:s + 500]]
    for k in range(4): cnt[k] += np.bincount(yy[:, k].ravel(), minlength=256)
json.dump(dict(consensus=cnt[0].tolist(), worldcover=cnt[1].tolist(), dynamicworld=cnt[2].tolist(), esri=cnt[3].tolist()),
          open(f"{OUT}/class_counts_test.json", "w"))
tab = {}
for k, nm in enumerate(["consensus", "worldcover", "dynworld", "esri"]):
    c = cnt[k]; kn = c[:9].sum()
    tab[nm] = [100 * c[q] / kn for q in range(9)] + [100 * c[253] / c.sum(), 100 * c[255] / c.sum()]
print(pd.DataFrame(tab, index=CLS + ["rangeland(253) % of all", "ignored % of all"]).round(1).to_string())
idx = np.sort(np.random.default_rng(0).choice(vi, min(100, len(vi)), replace=False))
xs = X[idx].astype(np.float32)
print("channel means (dB):", xs.mean((0, 2, 3)).round(2), "| NaN:", int(np.isnan(xs).sum()))
print("files:", sorted(os.listdir(OUT)))
print("total GB:", round(sum(os.path.getsize(f"{OUT}/{f}") for f in os.listdir(OUT) if os.path.isfile(f"{OUT}/{f}")) / 1e9, 2))

places: 1124 | dirs: 1124 | cons dir: /kaggle/input/notebooks/vinayybg/v2-01-consensus/cons
test places: 148 | usable (>=3 months): 147 | excluded: [594]
test 0 / 147 3s
test 20 / 147 17s
test 40 / 147 34s
test 60 / 147 49s
test 80 / 147 70s
test 100 / 147 89s
test 120 / 147 105s
test 140 / 147 120s

test: valid samples 4402 of 4410 | places 147 of 147
                         consensus  worldcover  dynworld  esri
water                          7.0         6.3       8.1   8.2
trees                         39.3        36.6      33.6  52.1
grass                         15.5        23.1       4.6   0.0
flooded                        1.8         2.3       1.0   4.1
crops                          9.1         8.6       9.7  14.2
shrub                          7.1         5.5      11.7   0.0
built                          4.6         2.7       4.1   7.0
bare                          13.9        14.2      17.5  12.1
snow                           1.6         0.7       9.7   2.4
rangeland(253) 